#3 - Data Integrity & Leakage-Free 5-Fold Cross-Validation Split


## Phase 1 Deliverables Overview
This notebook addresses all core requirements for **Phase 1: Prepare reliable evidence**:
1. **Source Identity & Duplicate Audit:** Identify duplicate pairs and consecutive exam frames.
2. **Leakage-Free Partitioning:** Generate a frozen 5-fold cross-validation split folds.csv using StratifiedGroupKFold.
3. **Anatomical Mask Verification:** Inspect `.mat` kidney masks and overlay them onto ultrasound images.
4. **Lesion Box Coordinate Verification:** Inspect bounding boxes annotations and overlay them on pathological images.

In [9]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the classification manifests y duplicate pairs of images
manifest_path = 'results/eda/classification_manifest.csv' if os.path.exists('results/eda/classification_manifest.csv') else 'classification_manifest.csv'
duplicates_path = 'results/eda/duplicate_pairs.csv'

df = pd.read_csv(manifest_path)
duplicates = pd.read_csv(duplicates_path)

df['filename'] = df['filename'].astype(str)

# 2. Filter only the real duplicate or the same exam ones from the csv
valid_leakage_pairs = duplicates[
    duplicates['verdict'].isin(['identical image', 'same exam (consecutive frame)'])
]

# 3. Construct the mapping groups
group_mapping = {}
group_counter = 0

for _, row in valid_leakage_pairs.iterrows():
    img_a = str(row['img_a']).strip()
    img_b = str(row['img_b']).strip()

    if not img_a.endswith('.jpg'):
        img_a += '.jpg'
    if not img_b.endswith('.jpg'):
        img_b += '.jpg'

    if img_a in group_mapping:
        gid = group_mapping[img_a]
    elif img_b in group_mapping:
        gid = group_mapping[img_b]
    else:
        gid = f"group_{group_counter}"
        group_counter += 1

    group_mapping[img_a] = gid
    group_mapping[img_b] = gid

# Assing group ID or mantain he archive name if its not a duplicate
df['group_id'] = df['filename'].map(group_mapping).fillna(df['filename'])

# 4. Generate Stratified Group 5-Fold Split
sgkf = StratifiedGroupKFold(n_splits=5)
df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df['diagnosis'], df['group_id'])):
    df.loc[val_idx, 'fold'] = fold

# 5. Save manifest frozen with the folds
os.makedirs('results/eda', exist_ok=True)
df.to_csv('results/eda/folds.csv', index=False)

grouped_count = df[df['group_id'].str.startswith('group_')]['group_id'].nunique()
print(f"Successfully created: 'results/eda/folds.csv'.  {grouped_count} clusters of images.")

Successfully created: 'results/eda/folds.csv'.  88 clusters of images.


## Validation & Audit Checks

Sanity checks executed:
1. **Class Balance:** Confirm healthy (0) vs pathological (1) target distribution across folds.
2. **Leakage Audit:** Verify zero `group_id` overlap between training and validation sets.

In [10]:
folds_df = pd.read_csv('results/eda/folds.csv')

print("CLASS DISTRIBUTION PER FOLD (%)")
print(pd.crosstab(folds_df['fold'], folds_df['diagnosis'], normalize='index') * 100)

leakage_found = False
for fold in range(5):
    val_groups = set(folds_df[folds_df['fold'] == fold]['group_id'])
    train_groups = set(folds_df[folds_df['fold'] != fold]['group_id'])

    overlap = val_groups.intersection(train_groups)
    if len(overlap) > 0:
        print(f" There is leakage in fold {fold}: {overlap}")
        leakage_found = True

if not leakage_found:
    print("\n AUDIT PASSED: Zero data leakage detected across all 5 folds.")

CLASS DISTRIBUTION PER FOLD (%)
diagnosis          0          1
fold                           
0          22.670025  77.329975
1          22.670025  77.329975
2          22.727273  77.272727
3          22.727273  77.272727
4          22.556391  77.443609

 AUDIT PASSED: Zero data leakage detected across all 5 folds.


##Verification of duplicates / Near-Duplicates

In [11]:
import matplotlib.pyplot as plt
import cv2

# Visualización de casos duplicados / mismo examen (Source identity verification)
if len(valid_leakage_pairs) > 0:
    sample_pair = valid_leakage_pairs.iloc[0]
    img_a_name = str(sample_pair['img_a']).strip()
    img_b_name = str(sample_pair['img_b']).strip()
    if not img_a_name.endswith('.jpg'): img_a_name += '.jpg'
    if not img_b_name.endswith('.jpg'): img_b_name += '.jpg'

    path_a = os.path.join('bbdd_kidney', 'images', img_a_name)
    path_b = os.path.join('bbdd_kidney', 'images', img_b_name)

    if os.path.exists(path_a) and os.path.exists(path_b):
        img_a = cv2.cvtColor(cv2.imread(path_a), cv2.COLOR_BGR2RGB)
        img_b = cv2.cvtColor(cv2.imread(path_b), cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(img_a)
        plt.title(f"Image A: {img_a_name}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(img_b)
        plt.title(f"Image B: {img_b_name}\nVerdict: {sample_pair['verdict']}")
        plt.axis('off')

        plt.suptitle("Near-Duplicate / Same Exam Case Verification", fontsize=12)
        plt.tight_layout()
        plt.show()

Inspection of .mat and Images

In [14]:
import scipy.io as sio
import matplotlib.patches as patches
import cv2
import os

# 1. Cargar una imagen patológica (diagnosis == 1) para inspeccionar la máscara .mat
pathology_df = df[df['diagnosis'] == 1]
sample_row = pathology_df.iloc[0] if len(pathology_df) > 0 else df.iloc[0]

img_name = str(sample_row['filename'])
if not img_name.endswith('.jpg'):
    img_name += '.jpg'

base_name = img_name.replace('.jpg', '')

img_path = os.path.join('bbdd_kidney', 'images', img_name)
mask_path = os.path.join('bbdd_kidney', 'masks', f"{base_name}.mat")

if os.path.exists(img_path) and os.path.exists(mask_path):
    # Cargar imagen
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Cargar datos del archivo .mat
    mat_data = sio.loadmat(mask_path)
    # Filtrar claves internas de MATLAB (que empiezan por '__')
    valid_keys = [k for k in mat_data.keys() if not k.startswith('__')]

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # 1 : Original Image
    ax[0].imshow(img_rgb)
    ax[0].set_title(f"Original Image: {img_name}\nDiagnosis: {sample_row['diagnosis']}")
    ax[0].axis('off')

    #2 : Masl Overlap / ANotation .mat
    ax[1].imshow(img_rgb)

    if len(valid_keys) > 0:
        mask_matrix = mat_data[valid_keys[0]]
        ax[1].imshow(mask_matrix, alpha=0.4, cmap='jet')
        ax[1].set_title(f"Anatomical Mask Overlay (.mat)\nKey: '{valid_keys[0]}'")
    else:
        ax[1].set_title("No mask array found inside .mat file")

    ax[1].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"Verified alignment for {img_name}. Available MATLAB structure keys: {valid_keys}")
else:
    print(f"Check path: Image or mask missing for {img_name} in 'bbdd_kidney/images' or 'bbdd_kidney/masks'")

⚠️ Check path: Image or mask missing for 1.jpg in 'bbdd_kidney/images' or 'bbdd_kidney/masks'
